## Setup Spark with Iceberg and MinIO
This configures a Spark session with Iceberg support and sets up the lake catalog to use MinIO as the warehouse (s3a://bronze/iceberg).
We're using HadoopCatalog and enabling S3A access with local MinIO credentials.

In [1]:
import os
from pyspark.sql import SparkSession

packages = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2", # Iceberg runtime for Spark 3.5
    "org.apache.hadoop:hadoop-aws:3.3.4",  # Hadoop support for AWS S3 (S3A)
    "com.amazonaws:aws-java-sdk-bundle:1.12.530",  # AWS SDK bundle (used under the hood)
])

spark = (
    SparkSession.builder
      .appName("Iceberg Lakehouse on MinIO")
      .config("spark.jars.packages", packages)
      .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") # Enable Iceberg SQL support
      .config("spark.sql.defaultCatalog", "lake") # Use 'lake' as the default catalog for Spark SQL

      # Define the 'lake' Iceberg catalog using HadoopCatalog backed by MinIO 
      .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
      .config("spark.sql.catalog.lake.type", "hadoop")
      .config("spark.sql.catalog.lake.warehouse", "s3a://bronze/iceberg")

      # MinIO (S3-compatible) connection configs
      .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
      .config("spark.hadoop.fs.s3a.path.style.access", "true") # Enable path-style access (important for MinIO)
      .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") # Disable SSL for local setup
      .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
      .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
      .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

      .getOrCreate()
)

print("Spark:", spark.version)

Spark: 3.5.3


## Create Iceberg Table from Existing Parquet Files

We switch to the lake catalog, create the `people` namespace and define a new Iceberg table `people.staff` by loading `data.parquet` from MinIO.
This creates a managed Iceberg table that supports versioning, time travel and schema evolution from the start.

In [2]:
spark.catalog.setCurrentCatalog("lake")
spark.sql("CREATE NAMESPACE IF NOT EXISTS people")

spark.sql("""
CREATE TABLE IF NOT EXISTS people.staff
USING iceberg
AS
SELECT * FROM parquet.`s3a://raw-data/data.parquet`
""")

DataFrame[]

## Preview a few rows from people.staff
Run a quick Spark SQL sanity check to verify the table is readable and the expected columns are present.

In [3]:
spark.sql("""
SELECT name, country, residence, role, date FROM people.staff limit 5;
""").show(truncate=True)

+---------------+--------+---------+---------------+----------+
|           name| country|residence|           role|      date|
+---------------+--------+---------+---------------+----------+
|wallace camargo|  brazil| portugal|  data engineer|2026-10-10|
| halyna halchuk| ukraine| portugal|tester engineer|2026-10-10|
|  pedro esteves|portugal| portugal|   data analyst|2026-10-10|
|   shalom andre|  angola| portugal|  data engineer|2026-10-10|
|  rafael arruda|  brazil| portugal|  data engineer|2026-10-10|
+---------------+--------+---------+---------------+----------+



## View Table History for people.staff
Iceberg keeps track of table version history, including metadata about schema changes, snapshot lineage, and when each version became active. In snapshot_id you can see your first snapshot id and file for it you can find s3://bronze/iceberg/people/staff/metadata/snap-{snapshot_id}-**.avro

In [4]:
spark.sql("""
SELECT * FROM people.staff.history ORDER BY made_current_at DESC;
""").show(truncate=False)


+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-10-10 09:02:02.825|7107164313496359822|6489393479375103098|true               |
|2026-10-10 09:01:27.773|6489393479375103098|7644628415770968391|true               |
|2026-10-10 09:01:24.651|7644628415770968391|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+



## Check Data Files in people.staff
This shows the list of files in the Iceberg table, including:

file_path: where each file is stored
record_count: how many rows in each file
file_size_in_bytes: file size
content: type of data (e.g. data, delete)


In [5]:
spark.sql("""
SELECT content, file_path, record_count, file_size_in_bytes
FROM people.staff.files;
""").show(truncate=False)


+-------+---------------------------------------------------------------------------------------------------+------------+------------------+
|content|file_path                                                                                          |record_count|file_size_in_bytes|
+-------+---------------------------------------------------------------------------------------------------+------------+------------------+
|0      |s3a://bronze/iceberg/people/staff/data/00000-5-995e21f8-da40-423b-a56b-654842304552-0-00001.parquet|6           |1963              |
|0      |s3a://bronze/iceberg/people/staff/data/00000-6-bd9b2d1d-4c1c-48aa-a4c8-3963ee824984-0-00001.parquet|6           |1963              |
|0      |s3a://bronze/iceberg/people/staff/data/00000-1-c11aa2ff-2b15-4085-a9ac-ad0890f25e0b-0-00001.parquet|9           |2051              |
+-------+---------------------------------------------------------------------------------------------------+------------+------------------+



## Simulate a New Batch Ingestion
Since `data.parquet` is a single small file, a new batch is simulated by inserting a deterministic
subset of it — every row where `country = 'brazil'` — into the `people.staff` table.
After the insert, the history table is queried again to show that a new snapshot has been added.

This demonstrates how Iceberg creates a new table version with each write, enabling time travel and version tracking.

In [6]:
spark.sql("""
INSERT INTO people.staff
SELECT * FROM parquet.`s3a://raw-data/data.parquet` WHERE country = 'brazil';
""")

spark.sql("""
SELECT * FROM people.staff.history ORDER BY made_current_at DESC;
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-10-10 09:02:50.978|3989149706735745746|7107164313496359822|true               |
|2026-10-10 09:02:02.825|7107164313496359822|6489393479375103098|true               |
|2026-10-10 09:01:27.773|6489393479375103098|7644628415770968391|true               |
|2026-10-10 09:01:24.651|7644628415770968391|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+



## Save Snapshot IDs for Time Travel
Two snapshot IDs are copied from the `people.staff.history` table above to use in the next steps:

In [7]:
# Replace with the actual snapshot_id values from the history query above
snapshot_id_1 = 'REPLACE_WITH_FIRST_SNAPSHOT_ID'
snapshot_id_2 = 'REPLACE_WITH_SECOND_SNAPSHOT_ID'

## Verify the Rollback
This query checks the people.staff.history table again to confirm that the rollback took effect.
A new snapshot should appear at the top, pointing back to the older snapshot ID.

In [8]:

spark.sql("""
SELECT * FROM people.staff.history ORDER BY made_current_at DESC;
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-10-10 09:02:50.978|3989149706735745746|7107164313496359822|true               |
|2026-10-10 09:02:02.825|7107164313496359822|6489393479375103098|true               |
|2026-10-10 09:01:27.773|6489393479375103098|7644628415770968391|true               |
|2026-10-10 09:01:24.651|7644628415770968391|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+



## Check Row Count After Rollback
This query returns the current number of rows in the people.staff table, after the rollback.
It should match the count from the older snapshot (before the extra batch was inserted).

In [9]:
spark.sql("""
SELECT count(*) FROM people.staff  ;
""").show(truncate=False)

+--------+
|count(1)|
+--------+
|27      |
+--------+



## Add and Populate a New Column
This sequence adds a new column role_group to the people.staff table, classifying staff based on role.
The column is added after role, then populated with a basic categorization logic:

'leadership' for tech lead

'analytics' for data analyst

'engineering' for everything else

Finally, the results are grouped by role_group to show the distribution:

In [10]:


spark.sql("""ALTER TABLE people.staff ADD COLUMN role_group STRING AFTER role;
""")

spark.sql("""-- update a derived field to prove reads see the new column
UPDATE people.staff
SET role_group = CASE
  WHEN role = 'tech lead' THEN 'leadership'
  WHEN role = 'data analyst' THEN 'analytics'
  ELSE 'engineering'
END
WHERE role IS NOT NULL;
""")

spark.sql("""SELECT role_group, COUNT(*) FROM people.staff GROUP BY role_group;
""").show(truncate=False)


+-----------+--------+
|role_group |count(1)|
+-----------+--------+
|leadership |4       |
|analytics  |1       |
|engineering|22      |
+-----------+--------+



## Evolve Iceberg Schema On Write with `accept-any-schema`

In Iceberg, new columns can be added not only through `ALTER TABLE`, but also automatically during writes.

To enable this behavior, we switch the table to flexible write mode:

- Drop the `role_group` column to simplify the schema before adding something new.
- Set the table property `'write.spark.accept-any-schema' = true` to allow appending data with previously unseen columns.
- Add a new derived numeric column `name_length` (the length of each person's `name`) directly during ingestion — no schema pre-declaration needed.
- Use `.option("mergeSchema", "true")` to tell Iceberg to evolve the schema automatically.

In [11]:
from pyspark.sql.functions import col, length as slen


spark.sql("ALTER TABLE lake.people.staff DROP COLUMN role_group")
spark.sql("""ALTER TABLE lake.people.staff SET TBLPROPERTIES (
  'write.spark.accept-any-schema'='true'
)
""")

# Load the raw parquet file
df = spark.read.parquet("s3a://raw-data/data.parquet")

# Compute name_length as the number of characters in the name
df_with_len = df.withColumn("name_length", slen(col("name")))

(df_with_len.writeTo("lake.people.staff")
  .option("mergeSchema", "true")  # Enables schema evolution on write
  .append())

spark.sql("DESCRIBE TABLE lake.people.staff;").show(30)

+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|       name|   string|   NULL|
|    country|   string|   NULL|
|  residence|   string|   NULL|
|       role|   string|   NULL|
|       date|   string|   NULL|
|name_length|      int|   NULL|
+-----------+---------+-------+



## Enable Country-Based Partitioning (Start of Partition Evolution Test)
This step adds a partitioning rule on `country`.
It's the first step in testing partition evolution — a unique feature of Apache Iceberg that allows changing partition strategies over time without rewriting existing data.

Note: the original version of this notebook (NYC taxi data) partitioned by `days(tpep_pickup_datetime)`. All rows in `data.parquet` share the same `date`, so a day-based partition would collapse into a single bucket here — `country` gives a more illustrative multi-partition example with this dataset.

In [12]:
spark.sql("ALTER TABLE people.staff ADD PARTITION FIELD country")


DataFrame[]

## Check Existing Partitions
This shows how the people.staff table is currently partitioned.

Right now, the result shows partition = {NULL} and spec_id = 0, which means all the data was written before partitioning was added. There are no partitions yet — everything is stored together.

This confirms the starting point before testing partition evolution.

In [13]:
spark.sql("SELECT partition, record_count, file_count, last_updated_snapshot_id  FROM people.staff.partitions").show(truncate=False)


+---------+------------+----------+------------------------+
|partition|record_count|file_count|last_updated_snapshot_id|
+---------+------------+----------+------------------------+
|{NULL}   |36          |2         |1005170468758697559     |
+---------+------------+----------+------------------------+



## Set Write Strategy for New Data
These commands configure how new data will be written into the people.staff table:

WRITE DISTRIBUTED BY PARTITION tells Iceberg to group data by partition when writing files (useful after enabling partitioning).

WRITE ORDERED BY date, role sorts rows within each partition by date and role.

This helps optimize file layout for better query performance.

In [14]:
spark.sql("ALTER TABLE people.staff WRITE DISTRIBUTED BY PARTITION")
spark.sql("ALTER TABLE people.staff WRITE ORDERED BY date, role")

DataFrame[]

## Rewrite Existing Data Files with New Partitioning
This command rewrites the existing data in the people.staff table using Iceberg’s rewrite_data_files procedure.

It applies the new country-based partitioning and improves file layout by:

Compacting small files (min-input-files = 1)

Targeting ~128MB file sizes

Allowing partial progress commits in case of failures

This step brings older unpartitioned data in line with the new table configuration.

In [15]:
spark.sql("""
  CALL lake.system.rewrite_data_files(
    table => 'people.staff',
    strategy => 'binpack',
    options => map(
      'min-input-files','1',                 -- be aggressive: rewrite even tiny groups
      'target-file-size-bytes','134217728',  -- ~128MB target size
      'partial-progress.enabled','true'      -- commit progress even if all tasks don't finish
    )
  )
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

## Check Updated Partitions After Rewrite
Running this query again now shows actual partition values instead of {NULL}.
This confirms that the rewrite_data_files command applied the new country-based partitioning to existing data.

You should now see multiple partitions, each representing a different `country` value.

In [16]:
spark.sql("SELECT partition, record_count, file_count, last_updated_snapshot_id  FROM people.staff.partitions").show(truncate=False)


+----------+------------+----------+------------------------+
|partition |record_count|file_count|last_updated_snapshot_id|
+----------+------------+----------+------------------------+
|{ukraine} |2           |1         |3794837503355461818     |
|{angola}  |2           |1         |3794837503355461818     |
|{portugal}|2           |1         |3794837503355461818     |
|{brazil}  |30          |4         |3794837503355461818     |
+----------+------------+----------+------------------------+



## Verify Rewrite in Table History
This query checks the people.staff.history table to confirm that the data rewrite created a new snapshot.
You should see a new entry at the top, showing when the rewrite was committed and linking it to a new snapshot ID.

In [17]:
spark.sql("""-- verify it took effect
SELECT * FROM people.staff.history ORDER BY made_current_at DESC;
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-10-10 09:02:59.536|3794837503355461818|1005170468758697559|true               |
|2026-10-10 09:02:56.446|1005170468758697559|6534328912281263720|true               |
|2026-10-10 09:02:53.813|6534328912281263720|3989149706735745746|true               |
|2026-10-10 09:02:50.978|3989149706735745746|7107164313496359822|true               |
|2026-10-10 09:02:02.825|7107164313496359822|6489393479375103098|true               |
|2026-10-10 09:01:27.773|6489393479375103098|7644628415770968391|true               |
|2026-10-10 09:01:24.651|7644628415770968391|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+

